In [1]:
# ==========================================
# 1. THE HASHLIB METHOD (FOR VERIFICATION)
# ==========================================
import hashlib

def hash_with_library(message):
    # Encode string to bytes
    encoded_message = message.encode()
    
    # Calculate MD5
    md5_hash = hashlib.md5(encoded_message).hexdigest()
    
    # Calculate SHA-1
    sha1_hash = hashlib.sha1(encoded_message).hexdigest()
    
    return md5_hash, sha1_hash

message = "Fr. CRCE Computer Engineering"
md5_lib, sha1_lib = hash_with_library(message)

print(f"Message: {message}")
print(f"MD5 (hashlib):  {md5_lib}")
print(f"SHA-1 (hashlib): {sha1_lib}")


Message: Fr. CRCE Computer Engineering
MD5 (hashlib):  c1014f1372ce7f6ecf6f577c695cc706
SHA-1 (hashlib): e07059e8344e54706270f69723f6f1abf17b3d4c


In [2]:
# ==========================================
# 2. MD5 FROM SCRATCH (NO BUILT-IN HASH LIBS)
# ==========================================
import math

# Bitwise left rotation function (crucial for MD5)
def left_rotate(x, amount):
    x &= 0xFFFFFFFF
    return ((x << amount) | (x >> (32 - amount))) & 0xFFFFFFFF

# Generate the K table using math.sin (as per MD5 RFC)
def generate_K():
    return [int(abs(math.sin(i + 1)) * (2 ** 32)) & 0xFFFFFFFF for i in range(64)]

def md5_scratch(message):
    # 1. Initialization (Magic Numbers)
    A = 0x67452301
    B = 0xEFCDAB89
    C = 0x98BADCFE
    D = 0x10325476
    
    # 2. Padding the message
    msg_bytes = bytearray(message, 'utf-8')
    orig_len_bits = (8 * len(msg_bytes)) & 0xFFFFFFFFFFFFFFFF
    msg_bytes.append(0x80) # Append the '1' bit (10000000 in binary)
    
    # Pad with zeros until length % 64 == 56
    while len(msg_bytes) % 64 != 56:
        msg_bytes.append(0x00)
        
    # Append the original length as a 64-bit little-endian integer
    msg_bytes += orig_len_bits.to_bytes(8, byteorder='little')
    
    # MD5 Shift amounts for the 4 rounds
    s = [7, 12, 17, 22] * 4 + [5, 9, 14, 20] * 4 + [4, 11, 16, 23] * 4 + [6, 10, 15, 21] * 4
    K = generate_K()
    
    # 3. Process the message in 64-byte chunks
    for chunk_offset in range(0, len(msg_bytes), 64):
        chunk = msg_bytes[chunk_offset:chunk_offset + 64]
        
        # Break chunk into 16 32-bit words (little-endian)
        w = [int.from_bytes(chunk[i:i+4], byteorder='little') for i in range(0, 64, 4)]
        
        a, b, c, d = A, B, C, D
        
        # 4. The Main Loop (64 operations)
        for i in range(64):
            if 0 <= i <= 15:
                f = (b & c) | (~b & d)
                g = i
            elif 16 <= i <= 31:
                f = (d & b) | (~d & c)
                g = (5 * i + 1) % 16
            elif 32 <= i <= 47:
                f = b ^ c ^ d
                g = (3 * i + 5) % 16
            elif 48 <= i <= 63:
                f = c ^ (b | ~d)
                g = (7 * i) % 16
            
            f = (f + a + K[i] + w[g]) & 0xFFFFFFFF
            a, d, c, b = d, c, b, (b + left_rotate(f, s[i])) & 0xFFFFFFFF
            
        # Add chunk results back to main state
        A = (A + a) & 0xFFFFFFFF
        B = (B + b) & 0xFFFFFFFF
        C = (C + c) & 0xFFFFFFFF
        D = (D + d) & 0xFFFFFFFF

    # 5. Output: Combine A, B, C, D into a 128-bit hex string (little-endian)
    res = sum(x << (32 * i) for i, x in enumerate([A, B, C, D]))
    return f"{res.to_bytes(16, byteorder='little').hex()}"

# Test it!
print("MD5 (From Scratch):", md5_scratch(message))
print("Match with Hashlib?", md5_scratch(message) == md5_lib)

MD5 (From Scratch): c1014f1372ce7f6ecf6f577c695cc706
Match with Hashlib? True


In [3]:
# ==========================================
# 3. SHA-1 FROM SCRATCH (NO BUILT-IN HASH LIBS)
# ==========================================

def left_rotate(n, b):
    """Left rotate a 32-bit integer n by b bits."""
    return ((n << b) | (n >> (32 - b))) & 0xFFFFFFFF

def sha1_scratch(message):
    # 1. Initialization (Magic Numbers for SHA-1)
    H0 = 0x67452301
    H1 = 0xEFCDAB89
    H2 = 0x98BADCFE
    H3 = 0x10325476
    H4 = 0xC3D2E1F0
    
    # 2. Pre-processing (Padding)
    msg_bytes = bytearray(message, 'utf-8')
    orig_len_bits = (8 * len(msg_bytes))
    
    msg_bytes.append(0x80) # Append the '1' bit (10000000)
    
    # Pad with zeros until length is 56 bytes (448 bits) modulo 64
    while len(msg_bytes) % 64 != 56:
        msg_bytes.append(0x00)
        
    # Append the original length as a 64-bit BIG-ENDIAN integer
    # (Notice: MD5 uses little-endian, SHA-1 uses big-endian!)
    msg_bytes += orig_len_bits.to_bytes(8, byteorder='big')
    
    # 3. Process the message in successive 64-byte chunks
    for chunk_offset in range(0, len(msg_bytes), 64):
        chunk = msg_bytes[chunk_offset:chunk_offset + 64]
        
        # Break chunk into sixteen 32-bit big-endian words w[i]
        w = [0] * 80
        for i in range(16):
            w[i] = int.from_bytes(chunk[i*4:(i+1)*4], byteorder='big')
            
        # Extend the sixteen 32-bit words into eighty 32-bit words
        for i in range(16, 80):
            w[i] = left_rotate(w[i-3] ^ w[i-8] ^ w[i-14] ^ w[i-16], 1)
            
        # Initialize hash value for this chunk
        a, b, c, d, e = H0, H1, H2, H3, H4
        
        # 4. The Main Loop (80 operations)
        for i in range(80):
            if 0 <= i <= 19:
                f = (b & c) | ((~b) & d)
                k = 0x5A827999
            elif 20 <= i <= 39:
                f = b ^ c ^ d
                k = 0x6ED9EBA1
            elif 40 <= i <= 59:
                f = (b & c) | (b & d) | (c & d)
                k = 0x8F1BBCDC
            elif 60 <= i <= 79:
                f = b ^ c ^ d
                k = 0xCA62C1D6
                
            # The SHA-1 compression function
            temp = (left_rotate(a, 5) + f + e + k + w[i]) & 0xFFFFFFFF
            e = d
            d = c
            c = left_rotate(b, 30)
            b = a
            a = temp
            
        # Add this chunk's hash to result so far
        H0 = (H0 + a) & 0xFFFFFFFF
        H1 = (H1 + b) & 0xFFFFFFFF
        H2 = (H2 + c) & 0xFFFFFFFF
        H3 = (H3 + d) & 0xFFFFFFFF
        H4 = (H4 + e) & 0xFFFFFFFF

    # 5. Output: Produce the final 160-bit hash (hex formatting)
    return f"{H0:08x}{H1:08x}{H2:08x}{H3:08x}{H4:08x}"

# Test it!
print("SHA-1 (From Scratch):", sha1_scratch(message))
print("Match with Hashlib?  ", sha1_scratch(message) == sha1_lib)

SHA-1 (From Scratch): e07059e8344e54706270f69723f6f1abf17b3d4c
Match with Hashlib?   True
